
# Lab 6 — Model Deployment with Flask REST API
Sikandar Hussain — 502808



## Part 1 — Model Loading & Sanity Check



### Task 1.1 – Load the saved pipelines using joblib, no retraining happens here


In [ ]:

import joblib
import pandas as pd
import numpy as np

titanic_model = joblib.load("Lab6_Sikandar_Hussain_502808/titanic_pipeline.joblib")
housing_model = joblib.load("Lab6_Sikandar_Hussain_502808/housing_pipeline.joblib")

print("titanic:", titanic_model)
print("\nhousing:", housing_model)
print("\nboth loaded, no retraining happened")


: 


### Task 1.2 – Manual prediction with a sample passenger


In [ ]:

from sklearn.preprocessing import OneHotEncoder

sample = {
    "pclass": 3,
    "sex": "male",
    "age": 22,
    "sibsp": 1,
    "parch": 0,
    "fare": 7.25,
    "embarked": "S"
}

# add extra features that the pipeline was trained on
sample["adult_male"] = (sample["sex"] == "male") and (sample["age"] >= 18)
sample["alone"] = (sample["sibsp"] + sample["parch"]) == 0
sample["deck"] = "__nan__"

# fix nan categories in ohe so it works with newer sklearn
ct = titanic_model.named_steps["preprocess"]
for name, trans, cols in ct.transformers_:
    if isinstance(trans, OneHotEncoder):
        for i, cats in enumerate(trans.categories_):
            fixed = []
            needs_fix = False
            for c in cats:
                if isinstance(c, float) and np.isnan(c):
                    fixed.append("__nan__")
                    needs_fix = True
                else:
                    fixed.append(c)
            if needs_fix:
                trans.categories_[i] = np.array(fixed, dtype=object)

df = pd.DataFrame([sample])
df["adult_male"] = df["adult_male"].astype(bool)
df["alone"] = df["alone"].astype(bool)
print(df)

pred = titanic_model.predict(df)
prob = titanic_model.predict_proba(df)

label = "survived" if pred[0] == 1 else "did not survive"
print(f"\nprediction: {pred[0]} ({label})")
print(f"probability -> died: {prob[0][0]:.4f}, survived: {prob[0][1]:.4f}")


In [ ]:

housing_sample = {
    "MedInc": 8.3252, "HouseAge": 41.0,
    "AveRooms": 6.984, "AveBedrms": 1.024,
    "Population": 322.0, "AveOccup": 2.556,
    "Latitude": 37.88, "Longitude": -122.23,
}

df_h = pd.DataFrame([housing_sample])
print(df_h)

price = housing_model.predict(df_h)
print(f"\npredicted house value: {price[0]:.4f} (x $100k)")



## Part 2 — Classification REST API

the flask app is in app.py. models load once at startup, /health returns status, /predict takes json and returns prediction. validation handles missing fields, bad json, and wrong types. extra fields get ignored.


In [ ]:

import subprocess, time, requests

server = subprocess.Popen(
    ["python", "Lab6_Sikandar_Hussain_502808/app.py"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
time.sleep(3)
print("server started, pid:", server.pid)


In [ ]:

# health check
r = requests.get("http://localhost:5000/health")
print(f"status: {r.status_code}, response: {r.json()}")


In [ ]:

# valid input
payload = {"pclass": 3, "sex": "male", "age": 22, "sibsp": 1, "parch": 0, "fare": 7.25, "embarked": "S"}
r = requests.post("http://localhost:5000/predict", json=payload)
print(f"valid input -> status: {r.status_code}, response: {r.json()}")


In [ ]:

# missing field (no fare)
payload_missing = {"pclass": 3, "sex": "male", "age": 22, "sibsp": 1, "parch": 0, "embarked": "S"}
r = requests.post("http://localhost:5000/predict", json=payload_missing)
print(f"missing fare -> status: {r.status_code} (expect 400), response: {r.json()}")


In [ ]:

# wrong type (age is string)
payload_bad = {"pclass": 3, "sex": "male", "age": "young", "sibsp": 1, "parch": 0, "fare": 7.25, "embarked": "S"}
r = requests.post("http://localhost:5000/predict", json=payload_bad)
print(f"wrong type -> status: {r.status_code} (expect 400), response: {r.json()}")



## Part 3 — Regression Deployment

same app.py serves /predict_price for housing. classification gives a label (0 or 1), regression gives a continuous value. the pipeline keeps the fitted scaler so raw input gets scaled properly.


In [ ]:

# valid housing input
h_payload = {
    "MedInc": 8.3252, "HouseAge": 41.0, "AveRooms": 6.984, "AveBedrms": 1.024,
    "Population": 322.0, "AveOccup": 2.556, "Latitude": 37.88, "Longitude": -122.23
}
r = requests.post("http://localhost:5000/predict_price", json=h_payload)
print(f"valid housing -> status: {r.status_code}, response: {r.json()}")


In [ ]:

# missing population field
h_incomplete = {
    "MedInc": 8.3252, "HouseAge": 41.0, "AveRooms": 6.984, "AveBedrms": 1.024,
    "AveOccup": 2.556, "Latitude": 37.88, "Longitude": -122.23
}
r = requests.post("http://localhost:5000/predict_price", json=h_incomplete)
print(f"missing field -> status: {r.status_code} (expect 400), response: {r.json()}")



## Part 4 — Robustness Enhancements

app.py handles: extra fields get dropped, strings in numeric fields return 400, and every request gets logged to stdout.


In [ ]:

# extra fields should be ignored
payload_extra = {
    "pclass": 1, "sex": "female", "age": 30, "sibsp": 0, "parch": 0,
    "fare": 100.0, "embarked": "C", "cabin": "B22", "ticket": "12345"
}
r = requests.post("http://localhost:5000/predict", json=payload_extra)
print(f"extra fields -> status: {r.status_code} (expect 200), response: {r.json()}")


In [ ]:

# string in numeric field
h_bad = {
    "MedInc": "high", "HouseAge": 41.0, "AveRooms": 6.984, "AveBedrms": 1.024,
    "Population": 322.0, "AveOccup": 2.556, "Latitude": 37.88, "Longitude": -122.23
}
r = requests.post("http://localhost:5000/predict_price", json=h_bad)
print(f"type error -> status: {r.status_code} (expect 400), response: {r.json()}")


In [ ]:

# advanced: probability output, custom threshold, api key
adv_payload = {
    "pclass": 1, "sex": "female", "age": 30, "sibsp": 0, "parch": 0,
    "fare": 100.0, "embarked": "C", "return_proba": True, "threshold": 0.8
}

r = requests.post("http://localhost:5000/predict", json=adv_payload,
                  headers={"X-API-Key": "lab6-secret-key"})
print(f"proba + threshold + valid key -> status: {r.status_code}, response: {r.json()}")

r2 = requests.post("http://localhost:5000/predict", json=adv_payload,
                   headers={"X-API-Key": "wrong-key"})
print(f"invalid key -> status: {r2.status_code} (expect 401), response: {r2.json()}")


In [ ]:

server.terminate()
server.wait()
print("server stopped")



## Part 5 — Analytical Reflection

**1. why must preprocessing be serialized with the model?**
the model expects data in the exact transformed form it was trained on. if we dont save the preprocessing steps (imputer fill values, scaler means/stds, encoder mappings) alongside the model, we'd have to manually recreate them which is error prone and fragile.

**2. what would happen if we saved only model weights?**
predictions would be garbage because raw input has different scales and encodings than what the model learned on. the coefficients only make sense in the transformed feature space.

**3. why load model at startup instead of inside the endpoint?**
loading from disk is slow. doing it every request would add latency and waste memory creating duplicate objects. loading once keeps one copy in memory for all requests.

**4. is /predict idempotent?**
yes. same input always gives same output, no state changes happen on the server side. its basically a pure function behind http.

**5. what breaks with 100 simultaneous users?**
flask dev server is single threaded so requests queue up and latency spikes. connections can timeout under load. the model itself is read only so no data races, but throughput is the bottleneck.

**6. what changes for production?**
use gunicorn with multiple workers, dockerize it, add proper auth and rate limiting, structured logging and monitoring, model versioning, and deploy behind nginx with kubernetes for autoscaling.
